# Project Data Mining: Conference Popularity & Association Pattern Mining (DBLP)
Pipeline: extraction → preprocessing → popularitas → transactional DB → Apriori/FP-Growth → association rules → performa → temporal.

**Sumber data:** https://dblp.org/xml/dblp.xml.gz (diunduh 19 November 2026).

In [ ]:
import gzip, os, re, time, unicodedata, csv, math
from collections import Counter
from itertools import combinations
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from lxml import etree
from mlxtend.frequent_patterns import apriori as ml_apriori, fpgrowth as ml_fpgrowth, association_rules as ml_rules
from scipy import sparse

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"; RAW.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "output"; OUT.mkdir(exist_ok=True)
Y0, Y1 = 2015, 2025
SUPPORTS = [0.001, 0.005, 0.01, 0.02]
MIN_CONF = 0.5
PERIODS = [(2015, 2017), (2018, 2020), (2021, 2023), (2024, 2025)]
plt.rcParams["figure.figsize"] = (9, 4.5)

## 1. Data Extraction
File `dblp.xml.gz` + `dblp.dtd` diletakkan di `data/raw/`. Hanya elemen `inproceedings` (konferensi & workshop) tahun 2015–2025 yang disimpan. Sel ini dilewati bila CSV sudah ada (ekstraksi ±10 menit).

In [ ]:
CSV = RAW / "inproceedings_2015_2025.csv"
def extract(xml_gz, out_csv):
    os.chdir(xml_gz.parent)  # agar dblp.dtd ditemukan
    n_all = n_kept = 0
    with gzip.open(xml_gz, "rb") as src, open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["key","authors","title","year","booktitle","pages"])
        for _, el in etree.iterparse(src, events=("end",), tag="inproceedings",
                                     load_dtd=True, resolve_entities=True, huge_tree=True):
            n_all += 1
            year = (el.findtext("year") or "").strip()
            if year.isdigit() and Y0 <= int(year) <= Y1:
                t = el.find("title")
                w.writerow([el.get("key"), "|".join((a.text or "").strip() for a in el.findall("author")),
                            "".join(t.itertext()) if t is not None else "", year,
                            (el.findtext("booktitle") or "").strip(), (el.findtext("pages") or "").strip()])
                n_kept += 1
            el.clear()
            while el.getprevious() is not None: del el.getparent()[0]
    print(f"inproceedings total={n_all:,} | 2015-2025={n_kept:,}")
if not CSV.exists():
    cwd = Path.cwd(); extract(RAW / "dblp.xml.gz", CSV); os.chdir(cwd)
else:
    print("CSV sudah ada:", CSV)

## 2. Data Understanding

In [ ]:
raw = pd.read_csv(CSV, dtype=str, keep_default_na=False)
print("Jumlah record:", f"{len(raw):,}"); print("Atribut:", list(raw.columns))
print("Tipe key teratas:", raw.key.str.split("/").str[0].value_counts().to_dict())
raw.head()

## 3. Preprocessing
Langkah: (1) pilih konferensi/workshop (key `conf/*`), (2) buang missing values, (3) normalisasi author (hapus nomor disambiguasi DBLP, rapikan spasi/unicode, buang author ganda dalam satu paper), (4) normalisasi venue = akronim dari key (mis. `conf/aaai/..` → AAAI), (5) hapus duplikat (key sama, atau judul+tahun+venue sama), (6) pastikan `year` integer 2015–2025.

In [ ]:
log = {"data awal (2015-2025)": len(raw)}
df = raw.copy()
df["venue"] = df["key"].map(lambda k: k.split("/")[1].upper() if k.startswith("conf/") and k.count("/") >= 2 else None)
df = df[df.venue.notna()].copy(); log["setelah filter conf/workshop"] = len(df)
for c in ["authors","title","year","venue"]: df = df[df[c].str.strip() != ""]
log["setelah hapus missing"] = len(df)
df["is_workshop"] = df.booktitle.str.contains("workshop", case=False, na=False)
SUF = re.compile(r"\s+\d{4}$")
def norm_author(a): return re.sub(r"\s+", " ", SUF.sub("", unicodedata.normalize("NFKC", a))).strip()
def clean(s):
    seen, out = set(), []
    for a in s.split("|"):
        a = norm_author(a)
        if a and a not in seen: seen.add(a); out.append(a)
    return out
df["author_list"] = df.authors.map(clean); df = df[df.author_list.map(len) > 0]
df["year"] = df.year.astype(int); df = df[df.year.between(Y0, Y1)]
df["title_norm"] = df.title.str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
df = df.drop_duplicates("key").drop_duplicates(["title_norm","year","venue"]).reset_index(drop=True)
log["data akhir (setelah dedup)"] = len(df)
pd.Series(log).to_frame("jumlah record")

In [ ]:
print("Total publikasi    :", f"{len(df):,}")
print("Total author unik  :", f"{len({a for l in df.author_list for a in l}):,}")
print("Total venue unik   :", df.venue.nunique())
print("Workshop (booktitle mengandung 'workshop'):", f"{df.is_workshop.sum():,}")
by_year = df.groupby("year").size(); by_conf = df.groupby("venue").size().sort_values(ascending=False)
by_year.plot.bar(color="#3b6ea5", title="Publikasi per tahun"); plt.ylabel("publikasi"); plt.show()
by_conf.head(30).iloc[::-1].plot.barh(color="#3b6ea5", figsize=(8,8), title="30 venue terbanyak"); plt.show()
by_conf.head(10)

## 4. Conference Popularity Analysis (Task 2)

In [ ]:
pubs = df.groupby("venue").size().rename("n_pubs")
auth = df[["venue","author_list"]].explode("author_list").groupby("venue").author_list.nunique().rename("n_authors")
t = pd.concat([pubs, auth], axis=1).sort_values("n_pubs", ascending=False)
t["rank_pubs"] = t.n_pubs.rank(ascending=False, method="min").astype(int)
t["rank_authors"] = t.n_authors.rank(ascending=False, method="min").astype(int)
top_p = t.head(10); top_a = t.sort_values("n_authors", ascending=False).head(10)
display(top_p); display(top_a)
print("Irisan top-10:", sorted(set(top_p.index) & set(top_a.index)))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
top_p.iloc[::-1].n_pubs.plot.barh(ax=ax[0], color="#3b6ea5", title="Top 10 publikasi")
top_a.iloc[::-1].n_authors.plot.barh(ax=ax[1], color="#d98a2b", title="Top 10 author unik"); plt.tight_layout(); plt.show()
pv = df[df.venue.isin(top_p.index)].pivot_table(index="year", columns="venue", values="key", aggfunc="count", fill_value=0)
pv.plot(marker="o", figsize=(9,5), title="Tren publikasi 10 konferensi teratas"); plt.ylabel("publikasi"); plt.show()
t.to_csv(OUT/"t2_venue_ranking.csv")

## 5. Transactional Database (Task 3)

In [ ]:
ab, ay = {}, {}
for authors, v, y in zip(df.author_list, df.venue, df.year):
    for a in authors:
        ab.setdefault(a, set()).add(v); ay.setdefault((a, y), set()).add(v)
for name, d in [("Author-Based", ab), ("Author-Year-Based", ay)]:
    s = pd.Series([len(v) for v in d.values()])
    print(f"{name}: transaksi={len(d):,} | item unik={len({i for v in d.values() for i in v})} | "
          f"rata2 item/tx={s.mean():.2f} | median={s.median()} | max={s.max()}")
    print("  distribusi item/tx (1..10):", s.value_counts().sort_index().head(10).to_dict())
print("\nContoh Author-Based:")
display(pd.DataFrame([(k, ", ".join(sorted(v))) for k, v in list(ab.items())[:10]], columns=["author","venues"]))
print("Contoh Author-Year-Based:")
display(pd.DataFrame([(f"{k[0]} ({k[1]})", ", ".join(sorted(v))) for k, v in list(ay.items())[:10]], columns=["author-year","venues"]))

## 6. Algoritma Frequent Pattern Mining
Algoritma utama memakai **library mlxtend** (Apriori & FP-Growth). Sebagai pembanding/validasi, notebook juga menyertakan implementasi sendiri (sel di bawah). Transaksi di-encode ke matriks sparse boolean; hanya item yang memenuhi min-support yang dipertahankan (aman karena sifat anti-monotone).

In [ ]:
# --- Implementasi sendiri (Apriori & FP-Growth & rules) ---
from collections import Counter, defaultdict
from itertools import combinations
import math


def _weighted(transactions):
    """Gabungkan transaksi identik -> {frozenset: bobot}. Mempercepat counting."""
    c = Counter(frozenset(t) for t in transactions)
    return c


def min_count_from_support(min_support, n):
    return max(1, math.ceil(min_support * n - 1e-9))


# ----------------------------------------------------------------- Apriori
def apriori(transactions, min_support):
    """Return (itemsets: {frozenset: count}, n_transactions)."""
    wt = _weighted(transactions)
    n = sum(wt.values())
    min_count = min_count_from_support(min_support, n)

    item_cnt = Counter()
    for t, w in wt.items():
        for i in t:
            item_cnt[i] += w
    freq = {frozenset([i]): c for i, c in item_cnt.items() if c >= min_count}
    result = dict(freq)
    frequent_items = {next(iter(s)) for s in freq}
    # buang item tidak frequent dari transaksi (aman: anti-monotone)
    wt = _reduce(wt, frequent_items)

    prev = sorted(tuple(sorted(s)) for s in freq)
    k = 2
    while prev:
        cands = _gen_candidates(prev, k)
        if not cands:
            break
        cand_set = set(cands)
        counts = Counter()
        for t, w in wt.items():
            if len(t) < k:
                continue
            ts = sorted(t)
            if len(ts) < 40 or math.comb(len(ts), k) <= len(cand_set):
                for comb in combinations(ts, k):
                    if comb in cand_set:
                        counts[comb] += w
            else:  # transaksi besar: cek kandidat langsung
                tset = set(ts)
                for c in cand_set:
                    if tset.issuperset(c):
                        counts[c] += w
        cur = {c: v for c, v in counts.items() if v >= min_count}
        for c, v in cur.items():
            result[frozenset(c)] = v
        prev = sorted(cur)
        k += 1
    return result, n


def _reduce(wt, keep):
    out = Counter()
    for t, w in wt.items():
        r = t & keep
        if r:
            out[frozenset(r)] += w
    return out


def _gen_candidates(prev, k):
    """Join langkah Apriori (prefix k-2 sama) + prune subset."""
    prev_set = set(prev)
    cands = []
    for i in range(len(prev)):
        a = prev[i]
        for j in range(i + 1, len(prev)):
            b = prev[j]
            if a[:-1] != b[:-1]:
                break
            c = a + (b[-1],)
            if all(sub in prev_set for sub in combinations(c, k - 1)):
                cands.append(c)
    return cands


# --------------------------------------------------------------- FP-Growth
class _Node:
    __slots__ = ("item", "count", "parent", "children", "link")

    def __init__(self, item, parent):
        self.item, self.count, self.parent = item, 0, parent
        self.children, self.link = {}, None


def _build_tree(weighted_txns, order):
    root = _Node(None, None)
    header = {}
    tails = {}
    for items, w in weighted_txns:
        node = root
        for it in sorted((i for i in items if i in order), key=order.get):
            ch = node.children.get(it)
            if ch is None:
                ch = _Node(it, node)
                node.children[it] = ch
                if it in header:
                    tails[it].link = ch
                else:
                    header[it] = ch
                tails[it] = ch
            ch.count += w
            node = ch
    return header


def _mine(header, counts, min_count, suffix, out):
    # proses item dari frekuensi terendah
    for item in sorted(counts, key=lambda i: (counts[i], str(i))):
        new = suffix | {item}
        out[frozenset(new)] = counts[item]
        # conditional pattern base
        base = []
        node = header[item]
        while node is not None:
            path = []
            p = node.parent
            while p is not None and p.item is not None:
                path.append(p.item)
                p = p.parent
            if path:
                base.append((path, node.count))
            node = node.link
        cc = Counter()
        for path, w in base:
            for i in path:
                cc[i] += w
        cc = {i: c for i, c in cc.items() if c >= min_count}
        if not cc:
            continue
        order = {i: r for r, i in enumerate(sorted(cc, key=lambda x: (-cc[x], str(x))))}
        sub = _build_tree(((p, w) for p, w in base), order)
        _mine(sub, cc, min_count, new, out)


def fpgrowth(transactions, min_support):
    wt = _weighted(transactions)
    n = sum(wt.values())
    min_count = min_count_from_support(min_support, n)
    item_cnt = Counter()
    for t, w in wt.items():
        for i in t:
            item_cnt[i] += w
    item_cnt = {i: c for i, c in item_cnt.items() if c >= min_count}
    order = {i: r for r, i in enumerate(sorted(item_cnt, key=lambda x: (-item_cnt[x], str(x))))}
    header = _build_tree(wt.items(), order)
    out = {}
    _mine(header, item_cnt, min_count, frozenset(), out)
    return out, n


# ------------------------------------------------------- Association Rules
def association_rules(itemsets, n, min_confidence=0.5):
    """itemsets: {frozenset: count} (harus closed under subset = hasil Apriori/FPG)."""
    rows = []
    for itemset, cnt in itemsets.items():
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for ante in combinations(items, r):
                ante = frozenset(ante)
                cons = itemset - ante
                ca, cc = itemsets.get(ante), itemsets.get(cons)
                if not ca or not cc:
                    continue
                conf = cnt / ca
                if conf + 1e-12 >= min_confidence:
                    rows.append({
                        "antecedent": tuple(sorted(ante)),
                        "consequent": tuple(sorted(cons)),
                        "support": cnt / n,
                        "confidence": conf,
                        "lift": conf / (cc / n),
                        "count": cnt,
                    })
    return rows


In [ ]:
def encode(tx_list, min_support):
    """Transaksi -> DataFrame sparse boolean, hanya item frequent."""
    n = len(tx_list); cnt = Counter(i for t in tx_list for i in t)
    items = sorted(i for i, c in cnt.items() if c >= math.ceil(min_support * n - 1e-9))
    idx = {i: j for j, i in enumerate(items)}
    rows, cols = [], []
    for r, t in enumerate(tx_list):
        for i in t:
            if i in idx: rows.append(r); cols.append(idx[i])
    m = sparse.csr_matrix((np.ones(len(rows), dtype=bool), (rows, cols)), shape=(n, len(items)))
    return pd.DataFrame.sparse.from_spmatrix(m, columns=items), n

def run_lib(tx_list, sup, algo):
    X, n = encode(tx_list, sup)
    t0 = time.perf_counter()
    fi = (ml_apriori if algo == "apriori" else ml_fpgrowth)(X, min_support=sup, use_colnames=True)
    return fi, n, time.perf_counter() - t0

def lib_rules(fi, conf=MIN_CONF):
    if len(fi) == 0: return pd.DataFrame()
    return ml_rules(fi, metric="confidence", min_threshold=conf)

tx_author = list(ab.values())

## 7. Frequent Pattern Mining pada Author-Based DB (Task 4)

In [ ]:
results, store = [], {}
for sup in SUPPORTS:
    fa, n, ta = run_lib(tx_author, sup, "apriori")
    ff, _, tf = run_lib(tx_author, sup, "fpgrowth")
    ff["k"] = ff.itemsets.map(len); store[sup] = ff
    same = set(map(frozenset, fa.itemsets)) == set(map(frozenset, ff.itemsets))
    rules = lib_rules(ff)
    results.append({"min_support": sup, "freq_itemsets": len(ff), "k=1": (ff.k==1).sum(), "k=2": (ff.k==2).sum(),
                    "k=3": (ff.k==3).sum(), "rules": len(rules), "apriori_s": round(ta,2), "fpgrowth_s": round(tf,2), "hasil_sama": same})
    print(f"\n=== min_support {sup:.1%}: {len(ff)} frequent itemsets ===")
    display(ff.sort_values("support", ascending=False).head(10).assign(itemsets=lambda d: d.itemsets.map(lambda s: " + ".join(sorted(s)))))
perf = pd.DataFrame(results); perf

In [ ]:
print("Kombinasi konferensi (2-itemset) yang paling sering muncul bersama (min_support 0,1%):")
f = store[0.001]; display(f[f.k == 2].sort_values("support", ascending=False).head(15).assign(itemsets=lambda d: d.itemsets.map(lambda s: " + ".join(sorted(s)))))
perf.plot(x="min_support", y="freq_itemsets", logx=True, logy=True, marker="o", title="Jumlah frequent itemsets vs min_support"); plt.show()

## 8. Association Rule Mining (Task 5)
min confidence 50%; support, confidence, lift dihitung; 10 rule dengan lift tertinggi.

In [ ]:
rules_all = {}
for sup in SUPPORTS:
    r = lib_rules(store[sup]); rules_all[sup] = r
    if len(r) == 0: print(f"sup {sup:.1%}: tidak ada rule"); continue
    r = r.assign(antecedents=r.antecedents.map(lambda s: " + ".join(sorted(s))),
                 consequents=r.consequents.map(lambda s: " + ".join(sorted(s))))
    print(f"\n=== min_support {sup:.1%}: {len(r)} rules, top 10 lift ===")
    display(r.sort_values("lift", ascending=False).head(10)[["antecedents","consequents","support","confidence","lift"]])

**Support tinggi ≠ lift tinggi:** support mengukur seberapa sering, lift mengukur kekuatan asosiasi dibanding kejadian independen (lift>1 = positif). Konferensi yang sangat besar bisa punya support tinggi tetapi lift mendekati 1.

In [ ]:
r = rules_all[SUPPORTS[0]]
if len(r):
    pd.set_option("display.max_colwidth", 60)
    r2 = r.copy(); r2["n_items"] = r2.antecedents.map(len) + r2.consequents.map(len)
    plt.scatter(r2.support, r2.lift, c=r2.confidence, s=6, cmap="viridis"); plt.colorbar(label="confidence")
    plt.xscale("log"); plt.xlabel("support"); plt.ylabel("lift"); plt.title("Rules (min_support 0,1%)"); plt.show()

## 9. Perbandingan Performa (Task 6)
Waktu eksekusi Apriori vs FP-Growth (library) dan validasi terhadap implementasi sendiri.

In [ ]:
display(perf)
perf.plot(x="min_support", y=["apriori_s","fpgrowth_s"], logx=True, marker="o", title="Waktu eksekusi (detik)"); plt.show()
# Validasi implementasi sendiri vs library (min_support 1% dan 2% agar cepat)
for sup in [0.01, 0.02]:
    t0 = time.perf_counter(); own_a, n = apriori(tx_author, sup); ta = time.perf_counter() - t0
    t0 = time.perf_counter(); own_f, _ = fpgrowth(tx_author, sup); tf = time.perf_counter() - t0
    lib = {frozenset(s) for s in store[sup].itemsets}
    print(f"sup {sup:.0%}: own apriori={ta:.2f}s fpgrowth={tf:.2f}s | own==lib: {set(own_a)==lib and set(own_f)==lib}")

## 10. Temporal Association Analysis
Author-Year-Based DB per periode (min_support 0,5%, min confidence 50%).

In [ ]:
TS = 0.005; per_rules = {}
for a, b in PERIODS:
    tx = [v for (au, y), v in ay.items() if a <= y <= b]
    fi, n, _ = run_lib(tx, TS, "fpgrowth"); r = lib_rules(fi)
    per_rules[(a, b)] = r
    print(f"{a}-{b}: tx={n:,} | frequent itemsets={len(fi)} | rules={len(r)}")
keys = {p: {(tuple(sorted(x.antecedents)), tuple(sorted(x.consequents))) for x in r.itertuples()} if len(r) else set()
        for p, r in per_rules.items()}
stable = set.intersection(*keys.values())
print("\nRule yang bertahan di SEMUA periode:", len(stable), "| total rule unik:", len(set.union(*keys.values())))
for ant, con in sorted(stable)[:15]: print("  ", ant, "->", con)

## 11. Kesimpulan
*(Isi setelah hasil di atas dijalankan: jawab 7 pertanyaan kesimpulan pada bagian H soal berdasarkan angka hasil eksperimen.)*